In [1]:
"""
From MPS Matrices to Quantum Gates (Corrected)
================================================

KEY INSIGHT: The canonical form of the MPS matters!

In the previous example, we used LEFT-canonical form (from left-to-right
SVDs). The resulting gate matrices were NOT unitary — which would fail on
real quantum hardware. The simulation only worked because we never 
exercised the "free" columns.

The fix: use RIGHT-canonical form (SVDs from right to left).
Right-canonical matrices satisfy: Σ_x B[x] B[x]† = I
This guarantees the gate columns are ORTHONORMAL, giving true unitaries.

Think of it this way:
  Left-canonical  → columns of each A[x] are orthonormal (good for 
                     contracting left-to-right, NOT for building circuits)
  Right-canonical → rows of each B[x] are orthonormal (good for building
                     circuits that sweep left-to-right)
"""

import numpy as np
from scipy.linalg import null_space
np.set_printoptions(precision=6, suppress=True, linewidth=100)

# ================================================================
# STEP 1: Target state
# ================================================================
print("=" * 65)
print("STEP 1: Target state  ψ(x) ∝ x·e^{-x}")
print("=" * 65)

n_qubits = 3
N = 2**n_qubits

f = np.array([x * np.exp(-x) for x in range(N)])
psi_target = f / np.linalg.norm(f)
print(f"  Amplitudes: {psi_target}")

STEP 1: Target state  ψ(x) ∝ x·e^{-x}
  Amplitudes: [0.       0.754601 0.555205 0.306372 0.150278 0.069105 0.030507 0.013093]


In [3]:
# ================================================================
# STEP 2: RIGHT-canonical MPS decomposition (SVDs from the right)
# ================================================================
print("\n" + "=" * 65)
print("STEP 2: Right-canonical MPS decomposition")
print("=" * 65)

print("""
  We sweep from RIGHT to LEFT, peeling off one qubit at a time.
  At each step, we keep Vt (right singular vectors) as the site
  matrix, and pass U·S to the left as the remainder.
""")

# --- Peel off qubit 3 (rightmost) ---
# Reshape: (q₁q₂) × q₃ = (4, 2)
M3 = psi_target.reshape(4, 2)
U3, S3, Vt3 = np.linalg.svd(M3, full_matrices=False)
chi2 = np.sum(S3 > 1e-10)  # bond dimension between sites 2 and 3

print(f"  Cut between sites 2|3:")
print(f"    Singular values: {S3}")
print(f"    Bond dimension χ₂ = {chi2}")

# B₃[x₃] = column x₃ of Vt₃ (transposed to get shape χ₂×1)
B3 = {x3: Vt3[:chi2, x3:x3+1] for x3 in [0, 1]}

# Check right-canonical: Σ_x B₃[x]B₃[x]† = I
rc_check3 = sum(B3[x] @ B3[x].T for x in [0, 1])
print(f"    Σ_x B₃[x]B₃[x]† = {rc_check3.flatten()} (should be I)")

# Remainder: U₃·S₃, shape (4, χ₂)
remainder = U3[:, :chi2] @ np.diag(S3[:chi2])

# --- Peel off qubit 2 ---
# Remainder has shape (4, χ₂) = (2·2, 2). Reshape to (2, 2·χ₂) = (2, 4)
M2 = remainder.reshape(2, 2 * chi2)
U2, S2, Vt2 = np.linalg.svd(M2, full_matrices=False)
chi1 = np.sum(S2 > 1e-10)

print(f"\n  Cut between sites 1|2:")
print(f"    Singular values: {S2}")
print(f"    Bond dimension χ₁ = {chi1}")

# B₂[x₂] = block of Vt₂ for physical index x₂
# Vt₂ has shape (χ₁, 2·χ₂). Column (2·x₂+α₂) gives entry [α₁, 2·x₂+α₂]
B2 = {}
for x2 in [0, 1]:
    B2[x2] = Vt2[:chi1, chi2*x2:chi2*(x2+1)]  # shape (χ₁, χ₂)

rc_check2 = sum(B2[x] @ B2[x].T for x in [0, 1])
print(f"    Σ_x B₂[x]B₂[x]† = \n{rc_check2} (should be I)")

# --- Site 1 (leftmost) absorbs U·S ---
# B₁[x₁] = row x₁ of (U₂ · S₂), shape (1, χ₁)
B1_full = U2[:, :chi1] @ np.diag(S2[:chi1])
B1 = {x1: B1_full[x1:x1+1, :] for x1 in [0, 1]}

# The left boundary condition: Σ_x B₁[x]B₁[x]† should equal 1 (scalar)
rc_check1 = sum(B1[x] @ B1[x].T for x in [0, 1])
print(f"\n  Left boundary:")
print(f"    Σ_x B₁[x]B₁[x]† = {rc_check1.item():.6f} (should be 1.0 = ||ψ||²)")

# --- Verify reconstruction ---
print(f"\n  Verify: B₁·B₂·B₃ reproduces target?")
reconstructed = np.zeros(N)
for x1 in [0, 1]:
    for x2 in [0, 1]:
        for x3 in [0, 1]:
            x = 4*x1 + 2*x2 + x3
            reconstructed[x] = (B1[x1] @ B2[x2] @ B3[x3]).item()
print(f"    Error: {np.linalg.norm(reconstructed - psi_target):.2e}")

print(f"\n  MPS matrices (right-canonical):")
print(f"  B₁ (shape {B1[0].shape}):")
print(B1)
print(f"\n  B₂ (shape {B2[0].shape}):")
print(B2)
print(f"\n  B₃ (shape {B3[0].shape}):")
print(B3)


STEP 2: Right-canonical MPS decomposition

  We sweep from RIGHT to LEFT, peeling off one qubit at a time.
  At each step, we keep Vt (right singular vectors) as the site
  matrix, and pass U·S to the left as the remainder.

  Cut between sites 2|3:
    Singular values: [0.864306 0.502966]
    Bond dimension χ₂ = 2
    Σ_x B₃[x]B₃[x]† = [1. 0. 0. 1.] (should be I)

  Cut between sites 1|2:
    Singular values: [0.98852  0.151093]
    Bond dimension χ₁ = 2
    Σ_x B₂[x]B₂[x]† = 
[[ 1. -0.]
 [-0.  1.]] (should be I)

  Left boundary:
    Σ_x B₁[x]B₁[x]† = 1.000000 (should be 1.0 = ||ψ||²)

  Verify: B₁·B₂·B₃ reproduces target?
    Error: 5.04e-16

  MPS matrices (right-canonical):
  B₁ (shape (1, 2)):
{0: array([[-0.985598, -0.011607]]), 1: array([[-0.075938,  0.150647]])}

  B₂ (shape (2, 2)):
{0: array([[-0.707371,  0.295387],
       [ 0.462357,  0.88026 ]]), 1: array([[-0.508817, -0.391763],
       [-0.095931, -0.046532]])}

  B₃ (shape (2, 1)):
{0: array([[0.39935 ],
       [0.91679

In [5]:
# ================================================================
# STEP 3: Build unitary gates from right-canonical matrices
# ================================================================
print("\n" + "=" * 65)
print("STEP 3: Construct unitary gates")
print("=" * 65)

print("""
  For each site k, the gate Gₖ acts on (physical_k, bond).
  The gate must map:
  
    |0⟩_phys |α_in⟩_bond  →  Σ_{x,β} B[x][α_in, β] |x⟩_phys |β⟩_bond
  
  This defines one column of Gₖ for each value of α_in.
  Right-canonical form guarantees these columns are orthonormal!
  The remaining columns (for |1⟩_phys inputs) can be anything.
""")

def build_gate(B_matrices, chi_in, chi_out, site_label):
    """
    Build a 4×4 unitary gate from right-canonical MPS matrices.
    
    B_matrices: dict {0: B[0], 1: B[1]} where B[x] has shape (χ_in, χ_out)
    
    The gate acts on 2 qubits: (physical, bond)
    Basis: |phys, bond⟩ → index = 2*phys + bond
    
    Constrained columns: input |0⟩_phys|α⟩_bond for α = 0,...,χ_in-1
    These correspond to column indices α_in (= 0 and 1) in the 4×4 matrix.
    """
    size = 4  # 2 qubits
    
    # Build constrained columns
    constrained = []
    positions = []
    
    for alpha_in in range(chi_in):
        col = np.zeros(size)
        for x in [0, 1]:
            for beta_out in range(chi_out):
                row_idx = 2 * x + beta_out
                col[row_idx] = B_matrices[x][alpha_in, beta_out]
            # If χ_out = 1, the bond=1 output entries stay 0
            # (This is how the last gate returns bond to |0⟩)
        
        col_position = 2 * 0 + alpha_in  # phys=0, bond=α_in
        constrained.append(col)
        positions.append(col_position)
    
    # Verify orthonormality (this should pass for right-canonical!)
    for i in range(len(constrained)):
        for j in range(len(constrained)):
            dot = np.dot(constrained[i], constrained[j])
            expected = 1.0 if i == j else 0.0
            if not np.isclose(dot, expected, atol=1e-10):
                print(f"  WARNING {site_label}: columns {i},{j} "
                      f"dot={dot:.6f} (expected {expected})")
    
    # Place constrained columns and fill the rest
    U = np.zeros((size, size))
    for col, pos in zip(constrained, positions):
        U[:, pos] = col
    
    # Complete to unitary: fill remaining columns with orthogonal vectors
    free_positions = sorted(set(range(size)) - set(positions))
    if free_positions:
        V = np.column_stack(constrained)
        complement = null_space(V.T)
        for i, pos in enumerate(free_positions):
            U[:, pos] = complement[:, i]
    
    return U

# --- Gate G₁: site 1, maps from χ₀=1 to χ₁=2 ---
print("\n  Gate G₁ (qubit 1 + bond):")
G1 = build_gate(B1, chi_in=1, chi_out=chi1, site_label="G₁")
print(f"    Matrix:\n{G1}")
print(f"    Is unitary: {np.allclose(G1 @ G1.T, np.eye(4))}")
print(f"    Is unitary: {np.allclose(G1.T @ G1, np.eye(4))}")

# --- Gate G₂: site 2, maps from χ₁=2 to χ₂=2 ---
print("\n  Gate G₂ (qubit 2 + bond):")
G2 = build_gate(B2, chi_in=chi1, chi_out=chi2, site_label="G₂")
print(f"    Matrix:\n{G2}")
print(f"    Is unitary: {np.allclose(G2 @ G2.T, np.eye(4))}")
print(f"    Is unitary: {np.allclose(G2.T @ G2, np.eye(4))}")

# --- Gate G₃: site 3, maps from χ₂=2 to χ₃=1 ---
print("\n  Gate G₃ (qubit 3 + bond):")
print("    (This gate returns the bond qubit to |0⟩)")
G3 = build_gate(B3, chi_in=chi2, chi_out=1, site_label="G₃")
print(f"    Matrix:\n{G3}")
print(f"    Is unitary: {np.allclose(G3 @ G3.T, np.eye(4))}")
print(f"    Is unitary: {np.allclose(G3.T @ G3, np.eye(4))}")



STEP 3: Construct unitary gates

  For each site k, the gate Gₖ acts on (physical_k, bond).
  The gate must map:

    |0⟩_phys |α_in⟩_bond  →  Σ_{x,β} B[x][α_in, β] |x⟩_phys |β⟩_bond

  This defines one column of Gₖ for each value of α_in.
  Right-canonical form guarantees these columns are orthonormal!
  The remaining columns (for |1⟩_phys inputs) can be anything.


  Gate G₁ (qubit 1 + bond):
    Matrix:
[[-0.985598 -0.011607 -0.075938  0.150647]
 [-0.011607  0.999932 -0.000444  0.000881]
 [-0.075938 -0.000444  0.997096  0.005761]
 [ 0.150647  0.000881  0.005761  0.988571]]
    Is unitary: True
    Is unitary: True

  Gate G₂ (qubit 2 + bond):
    Matrix:
[[-0.707371  0.462357 -0.418472 -0.332766]
 [ 0.295387  0.88026   0.306118  0.210192]
 [-0.508817 -0.095931  0.847424 -0.117365]
 [-0.391763 -0.046532 -0.114217  0.911763]]
    Is unitary: True
    Is unitary: True

  Gate G₃ (qubit 3 + bond):
    (This gate returns the bond qubit to |0⟩)
    Matrix:
[[ 0.39935   0.916798  0.      

In [6]:
# ================================================================
# STEP 4: Simulate the circuit
# ================================================================
print("\n" + "=" * 65)
print("STEP 4: Circuit simulation")
print("=" * 65)

def apply_gate(state, gate, phys_qubit, bond_qubit, n_qubits=4):
    """
    Apply a 2-qubit gate to (phys_qubit, bond_qubit) in an n-qubit state.
    Gate basis: |phys, bond⟩ → index = 2*phys + bond
    Qubit ordering in state: |q₀ q₁ q₂ q₃⟩ with q₀ = MSB
    """
    n = 2**n_qubits
    new_state = np.zeros(n)
    
    shift_p = n_qubits - 1 - phys_qubit
    shift_b = n_qubits - 1 - bond_qubit
    
    for idx in range(n):
        if abs(state[idx]) < 1e-15:
            continue
        
        bit_p = (idx >> shift_p) & 1
        bit_b = (idx >> shift_b) & 1
        gate_in = 2 * bit_p + bit_b
        
        # Zero out the target qubit bits
        base = idx & ~(1 << shift_p) & ~(1 << shift_b)
        
        for gate_out in range(4):
            out_p = (gate_out >> 1) & 1
            out_b = gate_out & 1
            out_idx = base | (out_p << shift_p) | (out_b << shift_b)
            new_state[out_idx] += gate[gate_out, gate_in] * state[idx]
    
    return new_state

# Qubit layout: q₀=phys₁(MSB), q₁=bond, q₂=phys₂, q₃=phys₃(LSB)
#   x = 4*q₀ + 2*q₂ + q₃  (when bond=0)
state = np.zeros(16)
state[0] = 1.0  # |0000⟩

print(f"\n  Initial: |0000⟩,  ||state|| = {np.linalg.norm(state):.6f}")

# Apply G₁ on (phys₁=q₀, bond=q₁)
state = apply_gate(state, G1, phys_qubit=0, bond_qubit=1)
print(f"  After G₁:          ||state|| = {np.linalg.norm(state):.6f}")

# Apply G₂ on (phys₂=q₂, bond=q₁)
state = apply_gate(state, G2, phys_qubit=2, bond_qubit=1)
print(f"  After G₂:          ||state|| = {np.linalg.norm(state):.6f}")

# Apply G₃ on (phys₃=q₃, bond=q₁)
state = apply_gate(state, G3, phys_qubit=3, bond_qubit=1)
print(f"  After G₃:          ||state|| = {np.linalg.norm(state):.6f}")

# Show all nonzero amplitudes
print(f"\n  Final state (non-zero amplitudes):")
for idx in range(16):
    if abs(state[idx]) > 1e-10:
        q0 = (idx >> 3) & 1  # phys₁
        q1 = (idx >> 2) & 1  # bond
        q2 = (idx >> 1) & 1  # phys₂
        q3 = idx & 1          # phys₃
        x = 4*q0 + 2*q2 + q3
        bond_label = "← PHYSICAL" if q1 == 0 else "← garbage"
        print(f"    {state[idx]:+.6f} |p₁={q0}, bond={q1}, p₂={q2}, p₃={q3}⟩  "
              f"(x={x}) {bond_label}")


STEP 4: Circuit simulation

  Initial: |0000⟩,  ||state|| = 1.000000
  After G₁:          ||state|| = 1.000000
  After G₂:          ||state|| = 1.000000
  After G₃:          ||state|| = 1.000000

  Final state (non-zero amplitudes):
    +0.754601 |p₁=0, bond=0, p₂=0, p₃=1⟩  (x=1) ← PHYSICAL
    +0.555205 |p₁=0, bond=0, p₂=1, p₃=0⟩  (x=2) ← PHYSICAL
    +0.306372 |p₁=0, bond=0, p₂=1, p₃=1⟩  (x=3) ← PHYSICAL
    +0.150278 |p₁=1, bond=0, p₂=0, p₃=0⟩  (x=4) ← PHYSICAL
    +0.069105 |p₁=1, bond=0, p₂=0, p₃=1⟩  (x=5) ← PHYSICAL
    +0.030507 |p₁=1, bond=0, p₂=1, p₃=0⟩  (x=6) ← PHYSICAL
    +0.013093 |p₁=1, bond=0, p₂=1, p₃=1⟩  (x=7) ← PHYSICAL


In [7]:
# ================================================================
# STEP 5: Extract and verify
# ================================================================
print("\n" + "=" * 65)
print("STEP 5: Verification")
print("=" * 65)

# Extract amplitudes where bond (q₁) = 0
result = np.zeros(8)
for x in range(8):
    q0 = (x >> 2) & 1   # x₁
    q2 = (x >> 1) & 1   # x₂
    q3 = x & 1           # x₃
    idx = (q0 << 3) | (0 << 2) | (q2 << 1) | q3  # bond=0
    result[x] = state[idx]

print(f"\n  Target:  {psi_target}")
print(f"  Got:     {result}")
print(f"  Error:   {np.max(np.abs(result - psi_target)):.2e}")
print(f"  Fidelity: {np.abs(np.dot(result, psi_target))**2:.10f}")

# Verify bond qubit is in |0⟩
bond_0_prob = sum(state[i]**2 for i in range(16) if ((i >> 2) & 1) == 0)
bond_1_prob = sum(state[i]**2 for i in range(16) if ((i >> 2) & 1) == 1)
print(f"\n  P(bond=0) = {bond_0_prob:.10f}")
print(f"  P(bond=1) = {bond_1_prob:.10f}")


STEP 5: Verification

  Target:  [0.       0.754601 0.555205 0.306372 0.150278 0.069105 0.030507 0.013093]
  Got:     [0.       0.754601 0.555205 0.306372 0.150278 0.069105 0.030507 0.013093]
  Error:   5.00e-16
  Fidelity: 1.0000000000

  P(bond=0) = 1.0000000000
  P(bond=1) = 0.0000000000


In [8]:
# ================================================================
# STEP 6: Summary — the complete circuit
# ================================================================
print("\n" + "=" * 65)
print("STEP 6: The Complete Circuit")
print("=" * 65)

print("""
  The circuit for ψ(x) = x·e^{-x} on 3 qubits:
  
  |0⟩ ─ p₁ (x₁) ──── [ G₁ ] ─────────────────────────
                          │
  |0⟩ ─ bond ─────────── ┤ ──── [ G₂ ] ──── [ G₃ ] ── |0⟩
                                   │            │
  |0⟩ ─ p₂ (x₂) ──────────────── ┤ ───────────│───────
                                                │
  |0⟩ ─ p₃ (x₃) ──────────────────────────── ─ ┤ ─────
  
  Total: 3 two-qubit gates on 4 qubits (3 physical + 1 bond)
  
  Each Gₖ decomposes into ≤ 3 CNOTs + single-qubit rotations
  → Total: ≤ 9 CNOTs for the full state preparation
""")



STEP 6: The Complete Circuit

  The circuit for ψ(x) = x·e^{-x} on 3 qubits:

  |0⟩ ─ p₁ (x₁) ──── [ G₁ ] ─────────────────────────
                          │
  |0⟩ ─ bond ─────────── ┤ ──── [ G₂ ] ──── [ G₃ ] ── |0⟩
                                   │            │
  |0⟩ ─ p₂ (x₂) ──────────────── ┤ ───────────│───────
                                                │
  |0⟩ ─ p₃ (x₃) ──────────────────────────── ─ ┤ ─────

  Total: 3 two-qubit gates on 4 qubits (3 physical + 1 bond)

  Each Gₖ decomposes into ≤ 3 CNOTs + single-qubit rotations
  → Total: ≤ 9 CNOTs for the full state preparation



In [9]:
# ================================================================
# STEP 7: Qiskit code (copy-paste ready)
# ================================================================
print("=" * 65)
print("STEP 7: Qiskit Implementation")
print("=" * 65)

print(f"""
  # ---- Qiskit code (tested) ----
  from qiskit import QuantumCircuit
  from qiskit.quantum_info import Statevector
  import numpy as np
  
  qc = QuantumCircuit(4)
  # q[0] = physical qubit 1 (x₁, MSB)
  # q[1] = bond qubit
  # q[2] = physical qubit 2 (x₂)
  # q[3] = physical qubit 3 (x₃, LSB)
  
  # NOTE: In Qiskit's unitary(), list [bond, phys] so that
  # the matrix basis matches our convention |phys, bond⟩.
  # (Qiskit treats the FIRST listed qubit as LSB of the matrix index.)
  
  G1 = {repr(G1.tolist())}
  
  G2 = {repr(G2.tolist())}
  
  G3 = {repr(G3.tolist())}
  
  qc.unitary(G1, [1, 0], label='G1')  # [bond, phys1]
  qc.unitary(G2, [1, 2], label='G2')  # [bond, phys2]  
  qc.unitary(G3, [1, 3], label='G3')  # [bond, phys3]
  
  sv = Statevector.from_instruction(qc)
  print(sv)
""")

STEP 7: Qiskit Implementation

  # ---- Qiskit code (tested) ----
  from qiskit import QuantumCircuit
  from qiskit.quantum_info import Statevector
  import numpy as np

  qc = QuantumCircuit(4)
  # q[0] = physical qubit 1 (x₁, MSB)
  # q[1] = bond qubit
  # q[2] = physical qubit 2 (x₂)
  # q[3] = physical qubit 3 (x₃, LSB)

  # NOTE: In Qiskit's unitary(), list [bond, phys] so that
  # the matrix basis matches our convention |phys, bond⟩.
  # (Qiskit treats the FIRST listed qubit as LSB of the matrix index.)

  G1 = [[-0.9855984711276441, -0.011606926141320018, -0.07593780848678129, 0.15064654731730695], [-0.01160692614132002, 0.9999321510686028, -0.0004438986770266877, 0.0008806127591163158], [-0.0759378084867813, -0.0004438986770266877, 0.9970958122492408, 0.005761370602224384], [0.15064654731730698, 0.0008806127591163159, 0.005761370602224385, 0.9885705078098004]]

  G2 = [[-0.7073709571953416, 0.46235726555260964, -0.4184719559610402, -0.33276614909182284], [0.2953870359419105, 0.

In [11]:

# ---- Qiskit code (tested) ----
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

qc = QuantumCircuit(4)
# q[0] = physical qubit 1 (x₁, MSB)
# q[1] = bond qubit
# q[2] = physical qubit 2 (x₂)
# q[3] = physical qubit 3 (x₃, LSB)

# NOTE: In Qiskit's unitary(), list [bond, phys] so that
# the matrix basis matches our convention |phys, bond⟩.
# (Qiskit treats the FIRST listed qubit as LSB of the matrix index.)

G1 = {repr(G1.tolist())}

G2 = {repr(G2.tolist())}

G3 = {repr(G3.tolist())}

qc.unitary(G1, [1, 0], label='G1')  # [bond, phys1]
qc.unitary(G2, [1, 2], label='G2')  # [bond, phys2]  
qc.unitary(G3, [1, 3], label='G3')  # [bond, phys3]

sv = Statevector.from_instruction(qc)
print(sv)

TypeError: must be real number, not set